# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

## GRUPO: NOMBRE DEL GRUPO

# Integrantes
Joaquín Venezia. LU: 1338/23
Simón Atías. LU: 526/23
Lucía Insua. LU: 580/22
Victoria Arce. LU: 553/23

### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

**Respuesta 1.a**

* **Rango:** $\mathrm{rank}(Q)=11$. Como $Q$ tiene 12 columnas, $\dim\ker Q = 12-11 = 1$, es decir, las soluciones no nulas de $Q\mathbf{m}=\mathbf{0}$ son $\mathbf{m}$ y sus múltiplos (única a menos de un factor de escala).
* **Cantidad mínima de puntos:** cada punto aporta 2 filas, y se necesitan al menos 11 filas independientes: $2n \ge 11 \Rightarrow n \ge 6$. Con 5 puntos hay solo 10 ecuaciones, que no alcanzan.
* **Condición:** contar filas es necesario pero no suficiente: las $2n$ filas deben tener rango 11, y eso depende de la posición de los puntos. Los puntos no pueden ser todos coplanares; deben estar en posición general (tampoco sobre una superficie crítica, como una cúbica alabeada que contenga al centro de la cámara).
  * *Caso coplanar.* Si todos los puntos cumplen $aU+bV+cW+d=0$, el vector $\boldsymbol{\pi}=(a,b,c,d)^T$ satisface $P_i^T\boldsymbol{\pi}=0$ para todo $i$. Entonces $(\boldsymbol{\pi},\mathbf{0},\mathbf{0})$, $(\mathbf{0},\boldsymbol{\pi},\mathbf{0})$ y $(\mathbf{0},\mathbf{0},\boldsymbol{\pi})$ pertenecen a $\ker Q$, que ya tiene dimensión $\ge 3$; sumando la solución verdadera $\mathbf{m}$ queda $\dim\ker Q\ge 4$ y $\mathrm{rank}(Q)\le 8$. Por ejemplo, con $W=0$ las tres columnas asociadas a $W$ son nulas (lo que solo da $\mathrm{rank}(Q)\le 9$) y el 8 aparece al agregar la solución verdadera. La solución deja de ser única.
* **Con datos reales:** el rango 11 vale para datos exactos. Con ruido de medición, $Q$ tiene rango numérico 12 (en `calib/3d` el menor valor singular es $\sigma_{12}\approx 0{,}008>0$) y no existe $\mathbf{m}\neq\mathbf{0}$ con $Q\mathbf{m}=\mathbf{0}$ exacto; por eso se resuelve la minimización de 1.b.

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 8.

**Respuesta 1.b**

Se busca $\min_{\|\mathbf{m}\|=1} \|Q\mathbf{m}\|^2$. Reescribiendo:

$$\|Q\mathbf{m}\|^2 = (Q\mathbf{m})^T(Q\mathbf{m}) = \mathbf{m}^T Q^T Q\,\mathbf{m} = \mathbf{m}^T S\,\mathbf{m}, \qquad S = Q^TQ.$$

**Propiedades de $S$.** Es simétrica, ya que $S^T = Q^T(Q^T)^T = S$, y semidefinida positiva, ya que $\mathbf{x}^TS\mathbf{x} = \|Q\mathbf{x}\|^2 \ge 0$. Por el teorema espectral, $S = V\Lambda V^T$, con $V$ ortogonal (columnas $\mathbf{v}_1,\dots,\mathbf{v}_{12}$ autovectores ortonormales) y $\Lambda = \mathrm{diag}(\lambda_1\ge\dots\ge\lambda_{12}\ge 0)$.

**Cambio de base.** Como las $\mathbf{v}_i$ son una base ortonormal, $\mathbf{m} = V\mathbf{c} = \sum_i c_i\mathbf{v}_i$. Al ser $V$ ortogonal conserva la norma:

$$\|\mathbf{m}\|^2 = \|\mathbf{c}\|^2 = \sum_i c_i^2 = 1, \qquad \mathbf{m}^TS\mathbf{m} = \mathbf{c}^T\Lambda\,\mathbf{c} = \sum_i \lambda_i c_i^2.$$

**Minimización.** Los pesos $c_i^2\ge 0$ suman 1, así que $\sum_i\lambda_i c_i^2$ es un promedio ponderado de los autovalores y no puede ser menor que el menor de ellos:

$$\sum_i \lambda_i c_i^2 \;\ge\; \lambda_{12}\sum_i c_i^2 = \lambda_{12}.$$

La igualdad se alcanza con $\mathbf{c} = \mathbf{e}_{12}$ (todo el peso en el menor autovalor), es decir, $\mathbf{m} = V\mathbf{e}_{12} = \mathbf{v}_{12}$. La solución es única (salvo signo) cuando $\lambda_{12}<\lambda_{11}$, es decir, cuando el menor autovalor es simple. Eso ocurre si $\mathrm{rank}(Q)=11$ con datos en posición general (1.a); si el menor autovalor fuera múltiple, cualquier vector unitario de ese autoespacio sería solución.

**Resultado.** $\mathbf{m}$ es el autovector de $Q^TQ$ asociado a su menor autovalor. Si $Q = U\Sigma V^T$ es su SVD, entonces $Q^TQ = V\Sigma^2V^T$, por lo que $\mathbf{m}$ es la última columna de $V$ (vector singular derecho asociado a $\sigma_{\min}$), y el valor mínimo es $\|Q\mathbf{m}\| = \sigma_{\min}$.

Observaciones: $\mathbf{m}$ queda definido a menos de signo (si $\mathbf{m}$ es solución, $-\mathbf{m}$ también), que se fija con un criterio físico (por ejemplo, que los puntos queden delante de la cámara). Con datos ideales $\sigma_{\min}=0$ y $Q\mathbf{m}=\mathbf{0}$ exacto; con ruido $\sigma_{\min}>0$ y $\mathbf{m}$ es la mejor solución en el sentido de mínimos cuadrados homogéneos. Luego $\mathbf{m}$ se reordena en la matriz de proyección $M\in\mathbb{R}^{3\times4}$.

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



In [ ]:
import os
import numpy as np
from alc import (obtenerQ3d, resolverDLT3D,
                 devolverMatricesCalibracion3d, verificarCalibracion3d)

np.set_printoptions(precision=3, suppress=True)

# Carpeta de datos: en el zip del template aparece como " calib" (con un espacio adelante)
nombre = [d for d in os.listdir(".") if d.strip() == "calib"][0]
carpeta = os.path.join(nombre, "3d")

# 12 puntos 3D (U, V, W) y 210 detecciones de esos mismos puntos: cada fila
# tiene los pares (u, v) intercalados (24 columnas).
objPts = np.loadtxt(os.path.join(carpeta, "p_W_corners.txt"), delimiter=",")   # 12 x 3
detecciones = np.loadtxt(os.path.join(carpeta, "detected_corners.txt"))        # 210 x 24

variantes = [("libre",            False, False),
             ("alfa_u = alfa_v",  True,  False),
             ("gamma = 0",        False, True),
             ("ambas",            True,  True)]

for fila in [0, 50, 100]:
    imgPts = detecciones[fila].reshape(12, 2)

    # DLT: Q (24 x 12) -> M (3 x 4)
    Q = obtenerQ3d(imgPts, objPts)
    M = resolverDLT3D(Q)

    print(f"===== Deteccion {fila} =====")
    for nombreVar, alfasIguales, gammaCero in variantes:
        R, t, K = devolverMatricesCalibracion3d(M, alfasIguales, gammaCero)
        chequeo = verificarCalibracion3d(M, R, t, K, imgPts, objPts)
        print(f"--- {nombreVar} ---")
        print("K =\n", K)
        print("R =\n", R)
        print("t =", t)
        print(f"error maximo de reproyeccion: {chequeo['error_reproyeccion']:.2f} px")
        if nombreVar == "libre":
            print(f"max|R R^T - I| = {chequeo['ortogonalidad']:.1e}, "
                  f"det(R) = {chequeo['det_R']:.3f}, "
                  f"puntos delante de la camara: {chequeo['delante_camara']}")


### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.


**Respuesta 3 (rango de $Q$).** $Q\in\mathbb{R}^{2n\times 9}$ y $\mathbf{h}$ está definido a menos de escala (8 grados de libertad), así que hace falta $\mathrm{rank}(Q)=8$: así $\dim\ker Q = 9-8=1$ y la solución es única salvo factor de escala. Cada punto aporta 2 ecuaciones, luego $n\ge 4$, y no puede haber 3 puntos colineales (ni en el tablero ni en la imagen). Acá $n=77$, así que $Q$ es de $154\times 9$.

**Cómo se obtienen las esquinas.** La carpeta no trae las coordenadas $(u,v)$, así que se detectan con código propio (`tp/esquinas.py`): las esquinas internas son puntos silla ($\det$ Hessiana de la imagen suavizada muy negativo); se buscan máximos locales con precisión subpíxel y se ordenan armando la grilla y quedándose con el bloque completo de $11\times 7$ centrado.

**Algoritmo.** (1) $\mathbf{H}_j$ por DLT con normalización de Hartley, (2) $A\mathbf{b}=0$ con 2 filas por vista, (3) $B=K^{-T}K^{-1}$ y Cholesky, (4) pose de cada vista con $\mathbf r_1=\lambda K^{-1}\mathbf c_1$, $\mathbf r_2=\lambda K^{-1}\mathbf c_2$, $\mathbf r_3=\mathbf r_1\times\mathbf r_2$, $\mathbf t=\lambda K^{-1}\mathbf c_3$.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from alc import (resolverH, armarA, calcularMatrizK2D, obtenerPose2D, esquinasTablero,
                 multiplicacionDeMatrices, traspuesta)

np.set_printoptions(precision=3, suppress=True)

# Puntos del tablero (mm): (U, V) = (30 i, 30 j), i = 0..10, j = 0..6, W = 0.
# Orden k = j*11 + i, igual al de las esquinas detectadas.
objPts = np.array([[30.0 * i, 30.0 * j] for j in range(7) for i in range(11)])

# Carpeta de datos: en el zip del template aparece como " calib" (con un espacio adelante)
nombre = [d for d in os.listdir(".") if d.strip() == "calib"][0]
carpeta = os.path.join(nombre, "coplanar")

# 1) Esquinas (u, v) de cada vista y homografia H_j por DLT (con normalizacion de Hartley)
Hlst = []
imgPtsLst = []
for k in range(1, 21):
    imagen = mpimg.imread(os.path.join(carpeta, f"Im_R_{k}.png"))
    imgPts = esquinasTablero(imagen)                          # 77 x 2
    imgPtsLst.append(imgPts)
    Hlst.append(resolverH(imgPts, objPts))

# 2) K a partir de las 20 homografias
A = armarA(Hlst)                  # 40 x 6
K = calcularMatrizK2D(A)
print("K =\n", K)

# 3) Pose de las vistas 1, 10 y 20 y error de reproyeccion
for k in [1, 10, 20]:
    R, t = obtenerPose2D(Hlst[k - 1], K)
    Rt = np.hstack([R, t.reshape(3, 1)])
    P = np.hstack([objPts, np.zeros((77, 1)), np.ones((77, 1))])   # (U, V, 0, 1)
    proy = multiplicacionDeMatrices(P, traspuesta(multiplicacionDeMatrices(K, Rt)))
    uv = proy[:, 0:2] / proy[:, 2:3]
    err = np.sqrt(np.sum((uv - imgPtsLst[k - 1]) ** 2, axis=1))
    print(f"\nIm_R_{k}: R =\n{R}\n t (mm) = {t}\n error de reproyeccion: medio {np.mean(err):.2f} px, max {np.max(err):.2f} px")


### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from alc import resolverH, inversa, proyectarPuntos, circuloEnPlano

# Fotos (carpeta futbol del template)
fotoGol = os.path.join("futbol", "test-gol-enzo.png")
fotoOffside = os.path.join("futbol", "test-fuera-de-juego.png")

# Sistema de la cancha (esquema estadio_azteca.png, 105 x 68 m): U = distancia a la
# linea de gol del arco que se ataca (U = 0 en esa linea), V = posicion a lo ancho
# (0 a 68; el centro del arco esta en V = 34). Medidas reglamentarias:
semiAreaChica = 18.32 / 2                   # el area chica es de 5,5 x 18,32 m
profAreaChica = 5.5
profAreaGrande = 16.5                       # el area grande es de 16,5 x 40,32 m
semiAreaGrande = 40.32 / 2
radioArco = 9.15                            # arco del area, centrado en el punto penal (11 m)
mitadCuerda = np.sqrt(radioArco ** 2 - (profAreaGrande - 11.0) ** 2)   # arco con la linea del area grande: V = 34 +- 7,31

# ---------------------------------------------------------------- a) y b): gol-enzo
# 6 correspondencias (pixel de la foto -> metros de la cancha). Los pixeles se leyeron en
# recortes ampliados de la foto. Izquierda de la foto = V chico (la eleccion es arbitraria;
# solo hay que ser consistente).
pxGol = np.array([[196, 171], [841, 190],      # area chica, esquinas sobre la linea de gol
                  [156, 231], [868, 251],      # area chica, esquinas del frente
                  [147, 406], [852, 422]],     # arco del area grande con la linea del area grande
                 dtype=float)
cancha = np.array([[0.0, 34 - semiAreaChica], [0.0, 34 + semiAreaChica],
                   [profAreaChica, 34 - semiAreaChica], [profAreaChica, 34 + semiAreaChica],
                   [profAreaGrande, 34 - mitadCuerda], [profAreaGrande, 34 + mitadCuerda]])

H = resolverH(pxGol, cancha)         # cancha (metros) -> foto (pixeles)
Hinv = inversa(H)                    # foto (pixeles) -> cancha (metros)
errores = np.sqrt(np.sum((proyectarPuntos(H, cancha) - pxGol) ** 2, axis=1))
print("Error de reproyeccion de los 6 puntos de referencia (px):", errores)

# a) Distancia de Enzo (#24) a la linea de gol: pie de apoyo en la foto -> cancha con H^-1.
pieEnzo = np.array([807.0, 546.0])
U_enzo, V_enzo = proyectarPuntos(Hinv, pieEnzo)[0]
print(f"a) Enzo: U = {U_enzo:.1f} m (distancia a la linea de gol), V = {V_enzo:.1f} m")

# b) Jugadores argentinos (camiseta oscura): pie en la foto (px). El de "oculto" esta tapado
# por otros jugadores: su pie es una estimacion.
argentinos = {"Enzo": pieEnzo,
              "izquierda": np.array([108.0, 541.0]),
              "centro-izq": np.array([242.0, 407.0]),
              "centro-der": np.array([380.0, 326.0]),
              "oculto (estimado)": np.array([386.0, 288.0]),
              "derecha": np.array([849.0, 345.0])}

def lineasModelo(H, color, ax):
    """Dibuja sobre la foto las lineas de la cancha (area chica, area grande, arco, linea de gol)."""
    segmentos = [[[0, 0], [0, 68]],
                 [[0, 34 - semiAreaGrande], [profAreaGrande, 34 - semiAreaGrande],
                  [profAreaGrande, 34 + semiAreaGrande], [0, 34 + semiAreaGrande]],
                 [[0, 34 - semiAreaChica], [profAreaChica, 34 - semiAreaChica],
                  [profAreaChica, 34 + semiAreaChica], [0, 34 + semiAreaChica]]]
    for seg in segmentos:
        q = proyectarPuntos(H, np.array(seg, dtype=float))
        ax.plot(q[:, 0], q[:, 1], color=color, lw=1)
    arco = circuloEnPlano((11.0, 34.0), radioArco, 200)
    arco = arco[arco[:, 0] >= profAreaGrande]          # solo la parte fuera del area
    q = proyectarPuntos(H, arco)
    ax.plot(q[:, 0], q[:, 1], color=color, lw=1)

imagen = mpimg.imread(fotoGol)
fig, ax = plt.subplots(figsize=(14, 9))
ax.imshow(imagen[:, :, 0:3])
lineasModelo(H, "red", ax)                       # chequeo visual: el modelo cae sobre las lineas reales
ax.scatter(pxGol[:, 0], pxGol[:, 1], c="cyan", s=20)
for nombre, pie in argentinos.items():
    centro = proyectarPuntos(Hinv, pie)[0]                       # pie -> cancha
    circulo = circuloEnPlano(centro, 1.0, 60)                    # circulo de 1 m de radio en la cancha
    q = proyectarPuntos(H, circulo)                              # cancha -> foto
    ax.plot(q[:, 0], q[:, 1], color="yellow", lw=1.5)
ax.set_xlim(0, imagen.shape[1])
ax.set_ylim(imagen.shape[0], 0)
ax.set_title("Ej. 4 a) y b): circulos de 1 m de radio sobre los argentinos (modelo de cancha en rojo)")
plt.tight_layout()
plt.show()


**Método (4.a y 4.b).** El piso es un plano, así que la foto y la cancha se relacionan por una homografía $\lambda\,(u,v,1)^T=\mathbf H\,(U,V,1)^T$. Con 6 correspondencias (esquinas del área chica y extremos del arco del área grande, cuyas coordenadas salen de las medidas reglamentarias de `estadio_azteca.png`) se arma el sistema $Q\mathbf h=\mathbf 0$ de la ecuación 11 y se resuelve con `resolverH` (DLT con normalización de Hartley). Para medir se usa $\mathbf H^{-1}$ (pixel → metros) y para dibujar $\mathbf H$ (metros → pixel); en ambos casos se divide por la tercera coordenada.

**Chequeos.** El error de reproyección de los 6 puntos es de 0,6 a 2,1 px, y el modelo de cancha dibujado en rojo cae sobre las líneas reales de la foto. Además, el punto del arco $(U,V)=(20{,}15;\,34)$, que no se usó para ajustar $\mathbf H$, cae sobre la línea blanca de la foto, a ≤ 3 px (`tp/test_ej4.py`).

**Respuesta 4.a.** Enzo (#24) estaba a **≈ 22 m** de la línea de gol ($U=22{,}2$ m, $V=39{,}7$ m) cuando pateó. Si se perturban 1 px las referencias y 3 px el pie, el resultado varía unos ±0,1 m. A eso se suma el error del modelo: el arco del área, que no se usó para ajustar $\mathbf H$ salvo en sus extremos, queda hasta 3 px (≈ 0,2 m) corrido en el medio, porque una homografía no modela la distorsión de la lente, y Enzo está más allá del arco. Elegir otro punto de contacto del pie con el piso suma otros ~0,5 m, así que conviene dar "unos 22 m".

**Respuesta 4.b.** Los 6 jugadores de camiseta oscura quedan marcados con un círculo de 1 m de radio sobre el piso (se ven como elipses por la perspectiva). El jugador del centro-izquierda del área chica tiene el pie tapado por otros: su círculo es una estimación.


In [ ]:
# ---------------------------------------------------------------- c) posible offside
# Esta foto mira al arco desde un costado: V crece hacia la camara y U hacia el centro de
# la cancha. 6 correspondencias: esquinas de las dos areas, extremo del arco y la esquina
# de la cancha (base de la bandera).
pxOff = np.array([[736, 316], [735, 757],        # area grande, esquinas del frente
                  [1230, 391], [1368, 576],      # area chica, esquinas del frente
                  [736, 557],                    # arco del area grande con la linea del area grande
                  [1305, 236]], dtype=float)     # esquina de la cancha (U = 0, V = 0)
canchaOff = np.array([[profAreaGrande, 34 - semiAreaGrande], [profAreaGrande, 34 + semiAreaGrande],
                      [profAreaChica, 34 - semiAreaChica], [profAreaChica, 34 + semiAreaChica],
                      [profAreaGrande, 34 + mitadCuerda],
                      [0.0, 0.0]])
H2 = resolverH(pxOff, canchaOff)
H2inv = inversa(H2)
errores2 = np.sqrt(np.sum((proyectarPuntos(H2, canchaOff) - pxOff) ** 2, axis=1))
print("Error de reproyeccion de los 6 puntos de referencia (px):", errores2)

# Pies (px) de los defensores (naranja) y de los atacantes (blanco) que importan.
defensores = {"naranja 1 (pie der.)": (822, 332), "naranja 1 (pie izq.)": (800, 325),
              "naranja 2": (771, 368), "naranja 3 (pie izq.)": (735, 540),
              "naranja 3 (pie der.)": (767, 536), "naranja 4 (pie izq.)": (693, 567),
              "naranja 4 (pie der.)": (816, 562), "naranja 5": (705, 483)}
atacantes = {"blanco 1": (793, 352), "blanco 2": (790, 468),
             "blanco 3 (#24)": (843, 530), "blanco 4 (#5)": (872, 505)}

# Ultimo defensor de campo (el arquero, de celeste, es el ultimo): el de menor U.
Udef = {k: proyectarPuntos(H2inv, np.array(v, dtype=float))[0][0] for k, v in defensores.items()}
ultimo = min(Udef, key=Udef.get)
U0 = Udef[ultimo]
print(f"Ultimo defensor: {ultimo}, U0 = {U0:.2f} m  (linea de offside: U = {U0:.2f})")
for nombre, pie in atacantes.items():
    U, V = proyectarPuntos(H2inv, np.array(pie, dtype=float))[0]
    print(f"{nombre}: U = {U:.2f} m -> {'ADELANTE de la linea' if U < U0 else 'detras de la linea'} "
          f"({U0 - U:+.2f} m)")

imagen2 = mpimg.imread(fotoOffside)
fig, ax = plt.subplots(figsize=(15, 10.5))
ax.imshow(imagen2[:, :, 0:3])
linea = proyectarPuntos(H2, np.array([[U0, 0.0], [U0, 68.0]]))   # recta paralela a la linea de fondo
ax.plot(linea[:, 0], linea[:, 1], color="red", lw=2, label=f"linea de offside (U = {U0:.2f} m)")
for pie in atacantes.values():
    ax.scatter(pie[0], pie[1], c="cyan", s=25)
for pie in defensores.values():
    ax.scatter(pie[0], pie[1], c="yellow", s=18)
ax.set_xlim(0, imagen2.shape[1])
ax.set_ylim(imagen2.shape[0], 0)
ax.legend()
ax.set_title("Ej. 4 c): linea de fuera de juego por el pie del ultimo defensor")
plt.tight_layout()
plt.show()


**Respuesta 4.c.** Con 6 correspondencias (esquinas del área grande y chica, extremo del arco y esquina de la cancha) se estima la $\mathbf H$ de esta foto (error de reproyección de 0,3 a 1,5 px). El arquero (de celeste) es el último adversario, así que el "último defensor" es el jugador naranja de menor $U$: su pie adelantado marca la línea de fuera de juego en $U_0\approx 14{,}4$ m. La recta $U=U_0$ es paralela a la línea de fondo y se proyecta a la foto con $\mathbf H$ (en rojo).

Atacantes (pies): el blanco **#5** queda **≈ 0,5 m más adelantado** que la línea ($U\approx13{,}9$ m, ventaja $0{,}48\pm0{,}10$ m al perturbar los píxeles), por lo que estaba en posición de **fuera de juego**. El #24 queda unos 0,1 m detrás de la línea (dentro del error), y los otros dos blancos están claramente detrás.

**Limitaciones.** La homografía vale solo para puntos del piso: se usaron los pies. En el reglamento cuentan también la cabeza y el cuerpo (no los brazos), que no están en el plano del piso, así que el resultado es una aproximación. La incertidumbre de 0,1 m es solo por error de marcado de pixeles; la elección del punto de contacto del pie suma más.


### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

In [ ]:
import os
import numpy as np
from alc import resolverH, calcularKRestringida, proyectarPuntos

np.set_printoptions(precision=3, suppress=True)

# 5 puntos del mundo (X, Y, Z) y su pixel (u_k, v_k) en 4 vistas
datos = np.loadtxt(os.path.join("salguero", "world_img_pts.csv"), delimiter=";", skiprows=1)
mundo = datos[:, 1:3]          # se ignora Z (los puntos del cordon estan a 0,1 m)
centro = (800.0, 600.0)        # supuesto: punto principal en el centro de la imagen de 1600 x 1200

# una homografia por vista, con Z = 0 para todos los puntos
Hlst = []
for k in range(4):
    pix = datos[:, 4 + 2 * k: 6 + 2 * k]
    H = resolverH(pix, mundo)
    Hlst.append(H)
    err = proyectarPuntos(H, mundo) - pix
    print("vista", k, "- error de reproyeccion de H por punto (px):",
          np.sqrt(err[:, 0] ** 2 + err[:, 1] ** 2))

# f con las 4 vistas juntas (gamma = 0, pixeles cuadrados, centro fijo)
K, residuo = calcularKRestringida(Hlst, centro)
print("\nK =\n", K)
print("residuo relativo de las ecuaciones:", residuo)

# dispersion: f estimada con cada vista por separado
print("f por vista:", [round(calcularKRestringida([H], centro)[0][0, 0]) for H in Hlst])


**Respuesta 5.a.**

**Método.** Se aplica el algoritmo de Zhang (Ej. 3) tomando $Z=0$ para los 5 puntos: se calcula una homografía $\mathbf H_k$ por vista con `resolverH` y se arma el sistema $A\mathbf b=0$ con `armarA`. Los puntos del cordón están en realidad a $Z=0{,}1$ m, así que esto introduce un error sistemático que tratamos como ruido (la cátedra confirmó que es una aproximación aceptable). El error de reproyección de cada $\mathbf H_k$ es de 0,6 a 13 px según el punto.

**Por qué hay que restringir $\mathbf K$.** Con $\mathbf K$ completa (5 parámetros) Zhang da valores absurdos con estos datos: $\alpha_v\approx 211$, $\gamma\approx 1319$, $c_u\approx 281$, $c_v\approx 196$. Las 4 vistas son casi la misma cámara (la tercera fila de las $\mathbf H_k$ es casi igual en todas), así que las ecuaciones son casi dependientes, y además el ruido de $Z$ pesa mucho. Entonces se agregan supuestos: $\gamma=0$, píxeles cuadrados ($\alpha_u=\alpha_v=f$) y punto principal en el centro de la imagen (1600 × 1200, es decir $(800, 600)$). Queda una sola incógnita, $f$.

**Cuenta.** Con esa $\mathbf K$, $\mathbf B=\mathbf K^{-T}\mathbf K^{-1}=\frac{1}{f^2}\begin{pmatrix}1&0&-c_u\\0&1&-c_v\\-c_u&-c_v&c_u^2+c_v^2+f^2\end{pmatrix}$, y su vector $\mathbf b=(B_{11},B_{12},B_{22},B_{13},B_{23},B_{33})$ se escribe $\mathbf b=x\,\mathbf p+\mathbf e$ con $x=1/f^2$, $\mathbf p=(1,0,1,-c_u,-c_v,c_u^2+c_v^2)$ y $\mathbf e=(0,0,0,0,0,1)$. El sistema $A\mathbf b=0$ queda $x\,(A\mathbf p)=-A\mathbf e$: un cuadrado mínimo de una incógnita, $x=-\dfrac{(A\mathbf p)\cdot(A\mathbf e)}{(A\mathbf p)\cdot(A\mathbf p)}$, y $f=1/\sqrt{x}$. Es la función `calcularKRestringida`.

**Resultado.** Con las 4 vistas, $f\approx 3497$ px:
$$\mathbf K\approx\begin{pmatrix}3497&0&800\\0&3497&600\\0&0&1\end{pmatrix}.$$

**Limitaciones (importante).**
* $c_u, c_v$ y $\alpha_u=\alpha_v$ son **supuestos**: estos datos (5 puntos × 4 vistas casi iguales) no los determinan.
* La estimación es poco precisa: por vista sola $f$ va de 3028 a 4040 px, con 3 vistas de 3107 a 3709 px, y el residuo relativo de las ecuaciones es 0,35. Tomamos $f\approx 3500$ con una incertidumbre de entre 10 y 15 %.
* Chequeo aparte (no forma parte del entregable): un ajuste no lineal de $f$ reproyectando los puntos con su $Z$ real, con el mismo centro fijo, da $f\approx 3384$ px, a 3 % de nuestro valor.
* Las poses que salen de estas $\mathbf H_k$ con esta $\mathbf K$ no son buenas (los vectores $\mathbf r_1,\mathbf r_2$ salen lejos de ser ortonormales por el sesgo de $Z$); para la pose del 5.b se usa otro conjunto de puntos con $Z=0$.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from alc import (resolverH, calcularKRestringida, obtenerPose2D, reproyectarPose,
                 calcularAx, traspuesta)

np.set_printoptions(precision=3, suppress=True)

# 51 puntos: 6 filas = (indice, X, Y, Z, u, v), una columna por punto. Todos con Z = 0.
D = np.loadtxt(os.path.join("salguero", "gs_10_09_point_correspondences_v5.txt"))
mundo = D[1:4, :].T            # 51 x 3 (X, Y, Z) en metros
pix = D[4:6, :].T              # 51 x 2 (u, v) en pixeles

# homografia del piso a la imagen, y f con la misma restriccion que en 5.a
# (gamma = 0, pixeles cuadrados, punto principal en el centro de la imagen)
H = resolverH(pix, mundo[:, :2])
K, residuo = calcularKRestringida([H], (800.0, 600.0))
print("K =\n", K, "\nresiduo relativo:", residuo)

# pose (R, t) de la camara respecto del sistema de la calle
R, t = obtenerPose2D(H, K)
print("R =\n", R, "\nt =", t)
print("posicion de la camara en el mundo (-R^T t):", -calcularAx(traspuesta(R), t))

# reproyeccion de los 51 puntos con K [R | t]
rep = reproyectarPose(K, R, t, mundo)
err = np.sqrt(np.sum((rep - pix) ** 2, axis=1))
print("error de reproyeccion (px): medio %.1f, maximo %.1f" % (err.mean(), err.max()))

img = mpimg.imread(os.path.join("salguero", "img_10_18_47_351.jpg"))
plt.figure(figsize=(14, 10))
plt.imshow(img)
plt.plot(pix[:, 0], pix[:, 1], "go", markersize=8, markerfacecolor="none", label="puntos medidos")
plt.plot(rep[:, 0], rep[:, 1], "r+", markersize=10, label="reproyeccion con la pose")
plt.legend()
plt.title("Ej. 5.b - reproyeccion de los 51 puntos")
plt.show()


**Respuesta 5.b.**

**Método.** Los 51 puntos están todos en el piso ($Z=0$), así que es un problema plano como el del Ej. 3: con `resolverH` se calcula la homografía $\mathbf H$ del piso a la imagen y con `obtenerPose2D` se descompone en $\mathbf R$ y $\mathbf t$ usando $\mathbf K$. Como $\mathbf K$ no es dato, se estima $f$ con los mismos supuestos que en el 5.a ($\gamma=0$, píxeles cuadrados, punto principal en $(800,600)$) pero usando $\mathbf H$ de esta imagen (`calcularKRestringida([H], (800, 600))`). Para reproyectar se usa $(u,v)\sim\mathbf K(\mathbf R\mathbf P+\mathbf t)$ (`reproyectarPose`).

**Resultado.**
* $f\approx 2000$ px (residuo relativo de las ecuaciones 0,014, mucho menor que el 0,35 del 5.a):
$$\mathbf K\approx\begin{pmatrix}2000&0&800\\0&2000&600\\0&0&1\end{pmatrix}.$$
* La cámara queda a unos 3,7 m de altura, unos 22 m antes del origen del sistema de la calle, mirando hacia $+Y$ (posición $-\mathbf R^T\mathbf t\approx(6{,}7;\,-22{,}0;\,3{,}7)$ m): es una pose razonable.
* Los puntos reproyectados caen sobre los medidos: error medio de unos 8 px y máximo de unos 26 px (en los puntos más cercanos a la cámara, sobre la senda peatonal de la derecha).

**Limitaciones.**
* **Este $f$ no coincide con el del 5.a** ($\approx 3500$). Con los 51 puntos, ajustando $f$ con la pose refinada (chequeo aparte con SciPy, no forma parte del entregable), el mínimo del error está claramente en $f\approx 2000$ (RMS de 7 px) y con $f=3500$ el error sube a unos 52 px. Los 5 puntos del 5.a, en cambio, preferían $f\approx 3000$–$3500$. No sabemos por qué: puede ser que el punto principal no esté en el centro, distorsión de la lente, o que las vistas no tengan la misma configuración de cámara. Como los 51 puntos cubren mucho más rango de profundidad que los 5 del 5.a, usamos $f$ de esta imagen para la pose.
* La pose sale de $\mathbf H$ (sin refinamiento no lineal) y el error de 8 px de media probablemente incluye distorsión de la lente, que el modelo pinhole no considera.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from alc import (resolverH, inversa, proyectarPuntos, velocidadMedia, velocidadesLocales)

# --- instantes: el nombre de cada captura es img_hh_mm_ss_ms.jpg ---
carpeta = [r for r, _, a in os.walk(".") if os.path.basename(r) == "gs_dataset"][0]
nombres = sorted(f for f in os.listdir(carpeta) if f.startswith("img_") and f.endswith(".jpg"))
T = []
for f in nombres:
    h, mi, s, ms = [int(x) for x in f[4:-4].split("_")]
    T.append(h * 3600 + mi * 60 + s + ms / 1000)
T = np.array(T) - T[0]                        # segundos desde el primer frame
print(len(nombres), "frames, de", T[0], "a", T[-1], "s; dt entre", np.diff(T).min(), "y", np.diff(T).max(), "s")

# --- pixel -> metros del piso: H de los 51 puntos (Z = 0), invertida ---
D = np.loadtxt(os.path.join("salguero", "gs_10_09_point_correspondences_v5.txt"))
H = resolverH(D[4:6, :].T, D[1:3, :].T)
G = inversa(H)

# --- trayectorias en pixeles de cada agente (ver tp/extraer_trayectorias.py) ---
tray = np.loadtxt(os.path.join("salguero", "trayectorias_pix.csv"))   # agente, frame, u, v
agentes = [(0, "peatona de buzo rojo", "tab:red"), (1, "taxi", "tab:orange"), (2, "camioneta", "tab:blue")]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for a, nombre, color in agentes:
    f = tray[tray[:, 0] == a]
    frames = f[:, 1].astype(int)
    P = proyectarPuntos(G, f[:, 2:4])               # posiciones (X, Y) en metros
    t = T[frames]
    v, (vx, vy) = velocidadMedia(t, P)
    err = np.sqrt((P[:, 0] - (np.mean(P[:, 0]) + vx * (t - np.mean(t)))) ** 2 +
                  (P[:, 1] - (np.mean(P[:, 1]) + vy * (t - np.mean(t)))) ** 2)
    print("%-22s %2d puntos, t = %.2f a %.2f s: v = %.2f m/s = %.1f km/h  (v_x, v_y) = (%.2f, %.2f)  desvio de la recta (rms) %.2f m"
          % (nombre, len(t), t[0], t[-1], v, 3.6 * v, vx, vy, np.sqrt(np.mean(err ** 2))))
    ax1.plot(t, 3.6 * velocidadesLocales(t, P, k=4), "o-", color=color, markersize=4, label=nombre)
    ax1.axhline(3.6 * v, color=color, linestyle="--", linewidth=1)
    ax2.plot(f[:, 2], f[:, 3], "o", color=color, markersize=4, label=nombre)
ax1.set_xlabel("tiempo (s)"); ax1.set_ylabel("velocidad (km/h)")
ax1.set_title("Velocidad en cada instante (ventana de 9 puntos); linea punteada = velocidad media")
ax1.legend(); ax1.grid(alpha=0.3)
ax2.imshow(mpimg.imread(os.path.join(carpeta, nombres[-1])))
ax2.set_title("Trayectorias medidas (pixeles) sobre el ultimo frame"); ax2.legend()
plt.tight_layout()
plt.show()


**Respuesta 5.c.**

**Método.** Las capturas de la carpeta `gs_dataset` (46 frames, de unos 5,2 s en total) son de la misma cámara fija que la imagen del 5.b, así que vale la misma $\mathbf H$ del piso. El instante de cada frame sale del nombre (`img_hh_mm_ss_ms.jpg`); los intervalos no son uniformes (de 0,03 a 0,25 s), por eso se usa siempre el tiempo real de cada captura y no un $\Delta t$ fijo.
1. En cada frame se mide el **punto de apoyo en el piso** de cada agente (ver abajo): pixel $(u,v)$.
2. Se lo pasa a metros del piso con $\mathbf H^{-1}$ (`inversa(H)` y `proyectarPuntos`): $(X,Y)$ de la calle.
3. **Velocidad:** pendiente de la recta de mínimos cuadrados de $X(t)$ e $Y(t)$, $v=\sqrt{v_X^2+v_Y^2}$ (`velocidadMedia`). Para el gráfico en el tiempo se usa la misma pendiente en una ventana de 9 puntos (`velocidadesLocales`): diferencias de a pares serían muy ruidosas (un error de 3 px en un auto lejano son varios metros).

**Puntos medidos** (con `tp/extraer_trayectorias.py`, que guarda `salguero/trayectorias_pix.csv`; es una herramienta de medición, como marcar a mano, y no forma parte de las funciones del entregable):
* Peatona: parte más baja de los zapatos rojos (máscara de color), 45 de los 46 frames.
* Taxi: centro de la base de la mancha en movimiento (resta con el fondo), 27 frames (2,0 a 5,0 s). Antes está muy lejos para medirlo bien.
* Camioneta: centro de la base de la mancha oscura, 17 frames (2,9 a 5,2 s). Antes se mezcla con los otros autos del fondo.

**Resultados.**

| agente | velocidad | dirección |
|---|---|---|
| peatona de buzo rojo | $1{,}36$ m/s $\approx$ **4,9 km/h** | cruza hacia $+X$ ($v_X=1{,}30$, $v_Y=0{,}42$ m/s) |
| taxi | $14{,}5$ m/s $\approx$ **52 km/h** | hacia la cámara, $-Y$ ($v_Y=-14{,}5$ m/s) |
| camioneta | $14{,}8$ m/s $\approx$ **53 km/h** | hacia la cámara, $-Y$ ($v_Y=-14{,}8$ m/s) |

La velocidad de la peatona es la de una caminata normal y la de los autos es razonable para una calle urbana; el taxi y la camioneta van en fila a velocidades parecidas.

**Limitaciones.**
* Las posiciones en píxeles salen de segmentación automática por color y por fondo; se verificaron mirando las superposiciones, pero tienen errores de unos pocos píxeles. A los autos lejanos (parte alta de la imagen) eso les mide varios metros: el desvío de la recta ajustada es de 0,9 a 1,0 m para los autos y de 0,18 m para la peatona. Por eso la velocidad instantánea de los autos oscila entre unos 40 y 62 km/h alrededor del valor medio.
* Para los autos el "punto de apoyo" es el centro de la base del recuadro, que incluye algo de sombra y no es exactamente el piso: un punto a unos 0,3 m de altura se proyecta con la cámara a 3,7 m de altura y estira las distancias del orden del 9 %. Estimamos la velocidad de los autos con una incertidumbre del orden de 10 % (cifra a ojo, no verificada).
* La $\mathbf H$ del 5.b tiene errores de 3 a 14 px en la zona de la senda peatonal, y mayores (hasta 23 px) en el borde derecho, cerca de la vereda. El error es parcialmente sistemático y se cancela al restar posiciones, pero no del todo.
* Usamos una sola $\mathbf H$ para todos los frames porque la cámara es fija en esta secuencia (el primer frame es la imagen del 5.b). No se usó $\mathbf K$ ni la pose: para medir sobre el piso alcanza con la homografía.


### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---